# 21cm Lightcone Visualization

This notebook visualizes the fiducial 21cm brightness temperature lightcone from 21cmFAST.

## Setup and Imports

In [1]:
import sys
import pickle
sys.path.append("../")
sys.path.append("../21cmfast_sim/")

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import colormaps as cms
import matplotlib.pylab as pylab

# Jupyter setup
%load_ext autoreload
%autoreload 2
%matplotlib inline

# Custom modules for plotting and survey analysis
from plot_params import params
from galaxy_survey import *

pylab.rcParams.update(params)
cols_default = plt.rcParams['axes.prop_cycle'].by_key()['color']

/projectnb/darkcosmo/dark_photon_project/dark_photons_radio_sky/notebooks_for_paper/../21cmfast_sim/galaxy_survey.py:646: SyntaxWarning: invalid escape sequence '\o'
  scales as $\omega^{-1}$.
/projectnb/darkcosmo/dark_photon_project/dark_photons_radio_sky/notebooks_for_paper/../physics.py:33: SyntaxWarning: invalid escape sequence '\ '
  """Speed of light in cm s\ :sup:`-1`\ ."""
/projectnb/darkcosmo/dark_photon_project/dark_photons_radio_sky/notebooks_for_paper/../physics.py:35: SyntaxWarning: invalid escape sequence '\ '
  """Boltzmann constant in eV K\ :sup:`-1`\ ."""
/projectnb/darkcosmo/dark_photon_project/dark_photons_radio_sky/notebooks_for_paper/../physics.py:41: SyntaxWarning: invalid escape sequence '\ '
  """Newton's Gravitational Constant in cm\ :sup:`3` g\ :sup:`-1` s\ :sup:`-2`\ ."""
/projectnb/darkcosmo/dark_photon_project/dark_photons_radio_sky/notebooks_for_paper/../physics.py:50: SyntaxWarning: invalid escape sequence '\ '
  """Thomson cross section in cm\ :sup:`2`\ 

In [ ]:
p

## Survey Configuration

Initialize the Roman Space Telescope survey with 21cmFAST lightcone data and foreground cleaning setup.

In [ ]:
# Data paths for 21cmFAST lightcone and associated files
cache_name = f'/projectnb/darkcosmo/dark_photon_project/21cmfast_cache/v4_lightcones/'
lc_name = "LightCone_z5.5_HIIDIM=200_BOXLEN=300.0_r3843290498042390.h5"  # Main lightcone file
lightconer_name = "lightconer_seed3843290498042390.pkl"  # Angular lightcone projection

# Initialize Roman Space Telescope survey object
roman = Survey("Roman", "Lya",  # Roman survey targeting Lyman-alpha emitters
        cache=cache_name, 
        lc_name=lc_name, 
        lightconer_name=lightconer_name, 
        nside=2048,  # HEALPix resolution
        foregrounds_path="/projectnb/darkcosmo/dark_photon_project/21cmfast_cache/pyilc_foregrounds_0.05mJy/all_foreground_maps.npy",
        use_pixel_ilc=True,  # Enable pixel-based internal linear combination cleaning, irrelevant here 
        )

## Lightcone Visualization

Create a polar plot showing the evolution of 21cm brightness temperature from $5 \leq z \leq 35$.

In [ ]:
# Custom colormap for Epoch of Reionization visualization
# EoR_color = mpl.colors.LinearSegmentedColormap.from_list('EoR', [
#     (0.0,  'yellow'),   
#     (0.25, 'orange'),
#     (0.5,  'red'),
#     (0.75, 'black'),    
#     (0.85, 'blue'),
#     (0.95, 'cyan'),
#     (1.0,  'cyan')      
# ])

# Load angular lightcone geometry for plotting
with open(cache_name+lightconer_name, "rb") as f:
    ang_lcn = pickle.load(f)

# Set up polar plot (radius = comoving distance, angle = sky position)
fig, ax = plt.subplots(1,1,figsize=[28, 7.0], constrained_layout=True,subplot_kw={'projection': 'polar'})

# Configure radial limits and coordinate system
ax.set_ylim(ang_lcn.lc_distances.min().value, ang_lcn.lc_distances.max().value)
ax.set_rorigin(0)

# Angular coordinates for the lightcone slice
lon = np.linspace(0, roman.box_size_radians, roman.dim)
lat = np.linspace(0, roman.box_size_radians, roman.dim)[::-1]  # Reverse for increasing X-values
ax.set_thetamax(lat.max() * 180/np.pi)

# Create coordinate grids for plotting
H, D = np.meshgrid(lat, ang_lcn.lc_distances.value)

# Set up radial grid lines at key redshifts
rgrid = np.arange(0, D.max(), roman.box_len)
rgrid = rgrid[rgrid > D.min()]
rgrid = np.append(rgrid, D.max())

# Convert distances to redshifts for grid labels
rs_array = np.flipud([z_at_value(roman.cosmo.comoving_distance, d*un.Mpc).value for d in ang_lcn.lc_distances.value])

plt.grid(True)

# Plot brightness temperature evolution
img = ax.pcolormesh(H, D, (1e14 * np.flip(roman.mgamma[:, 0].T))**2, edgecolors='face', cmap='plasma', norm="log",)

# Label grid with redshifts and configure axes
ax.set_rgrids(rgrid[::2], labels=[f"{z_at_value(roman.cosmo.comoving_distance, d*un.Mpc).value:.1f}" for d in rgrid[::2]])
ax.set_thetagrids([np.rad2deg(lon[0]), np.rad2deg(lon[lat.shape[0]//2]), np.rad2deg(lon[-1])])
ax.set_xlabel(r"$z$", labelpad=-125,)
ax.tick_params(axis='x', pad=9)

# Add colorbar
cbar = plt.colorbar(img, fraction=0.004, pad=0.008, aspect=10,)
# cax = plt.colorbar(img, orientation="horizontal")

cbar.set_label(r"$10^{28} m_{\gamma}^2$ [eV$^2$]", fontsize=12)
# plt.tight_layout()

plt.savefig("plots/lightcone.png",dpi=300, bbox_inches='tight')